In [ ]:
import numpy as np
from numpy import pi
import matplotlib.pyplot as plt
%matplotlib inline

# Вариант 6

### Задание

<!-- 1. Ознакомиться с теоретической частью. -->

2. Для заданных дискретных сигналов ( ) и y( ) (либо специально подобранных музыкальных файлов) выполнить следующие задания (в соответствии с заданным вариантом, выданным преподавателем):

    <!-- - реализовать операцию свертки двух сигналов; -->
    
    <!-- - реализовать операцию корреляции двух сигналов; -->
    
    - для сигналов x( ) и y( ), а также результата выполнения операции свертки выполнить прямое и обратное преобразование Фурье;
    
    <!-- - построить графики амплитудного и фазового спектра сигналов; -->
    
    <!-- - построить графики сигналов во временной области; -->
    
    <!-- - экспериментально проверить корректность схем вычисления свертки и корреляции (в соответствии с рисунками 15 и 16). -->

<!-- 3. Сравнить полученные результаты при разных значениях, оценить эффективность алгоритмов в зависимости от . -->

<!-- > Примечание. Прямое и обратное преобразования Фурье, вычисление операций свертки и корреляции должны быть представлены в двух вариантах – реализованные студентом самостоятельно и с использованием Open Source библиотек. -->

<!-- 4. Оформить итоговый отчет с указанием исходных данных, результатов работы, построенных графиков, выводами о результатах работы. -->


#### Графики

<!-- 1.	x(t)	x(t) -->
<!-- 2.	y(t)	y(t) -->
<!-- 3.	x(t)	ДПФ: амплитудный спектр -->
<!-- 4.	x(t)	ДПФ: фазовый спектр -->
<!-- 5.	x(t)	ОДПФ -->
<!-- 6.	x(t)	БПФ: амплитудный спектр -->
<!-- 7.	x(t)	БПФ: фазовый спектр -->
<!-- 8.	x(t)	ОБПФ -->
<!-- 9.	y(t)	ДПФ: амплитудный спектр -->
<!-- 10.	y(t)	ДПФ: фазовый спектр -->
<!-- 11.	y(t)	ОДПФ -->
<!-- 12.	y(t)	БПФ: амплитудный спектр -->
<!-- 13.	y(t)	БПФ: фазовый спектр -->
<!-- 14.	y(t)	ОБПФ -->
<!-- 15.	x(t) y(t)	Свертка -->
<!-- 16.	x(t) y(t)	Свертка через БПФ -->
<!-- 17.	x(t) y(t)	Корреляция -->
<!-- 18.	x(t) y(t)	Корреляция через БПФ -->
<!-- ##### С использованием Open Source библиотек.	 -->
<!-- 19.	x(t)	БПФ: амплитудный спектр -->
<!-- 20.	x(t)	БПФ: фазовый спектр -->
<!-- 21.	y(t)	БПФ: амплитудный спектр -->
<!-- 22.	y(t)	БПФ: фазовый спектр -->
<!-- 23.	x(t) y(t)	Свертка -->
<!-- 25.	x(t) y(t)	Корреляция -->

- x(t) = Орган С4

In [ ]:
A_x = [1, 0.5, 0.3, 0.1]
f0_x = 262 #Hz
h_x = [1, 2, 4, 8]
phi_x = 0

- y(t) = Орган E4

In [ ]:
A_y = [1, 0.4, 0.2, 0.1]
f0_y = 330 #Hz
h_y = [1, 2, 4, 8]
phi_y = 0

### Графики

In [ ]:
def plot(y, x, title: str|None = None):
    plt.figure(figsize=(4, 3))
    plt.plot(x, y)
    plt.title(title)
    plt.grid(True)
    plt.show() 

In [ ]:
F = max(f0_x, f0_y)
F_S = 2 * F * max(h_y + h_x)

SAMPLE_COUNT = 1024

In [ ]:
def s(t, A, h, f0, phi):
    assert(len(A) == len(h))
    return sum(
        A[i] * np.sin(2*pi * h[i] * f0 * t + phi)
        for i in range(0, len(A))
    )
x = lambda t: s(t, A_x, h_x, f0_x, phi_x)
y = lambda t: s(t, A_y, h_y, f0_y, phi_y)

t = np.linspace(0, 5 / F, SAMPLE_COUNT)

plot(x(t), t, "x(t)")
plot(y(t), t, "y(t)")

In [ ]:
def dft_mat(N: int):
    W = np.exp(-1j * 2*pi/N)
    return np.pow(W, np.fromfunction(lambda i,j: i*j, (N, N)))

def discrete_fourier_transform(x):    
    return dft_mat(len(x)) @ x

def inverse_discrete_fourier_transform(x):
    return np.linalg.inv(dft_mat(len(x))) @ x

t = np.linspace(0, SAMPLE_COUNT / F_S, SAMPLE_COUNT)
f = np.linspace(0, F_S, SAMPLE_COUNT)

dft_x = discrete_fourier_transform(x(t))
plot(np.abs(dft_x), f, "DFT[x(t)] (amplitude domain)")
plot(np.angle(dft_x), f, "DFT[x(t)] (phase domain)")
plot(
    np.real(inverse_discrete_fourier_transform(dft_x)) [:int(5/F * F_S)],
    t [:int(5/F * F_S)],
    "IDFT[DFT[x(t)]]"
)

dft_y = discrete_fourier_transform(y(t))
plot(np.abs(dft_y), f, "DFT[y(t)] (amplitude domain)")
plot(np.angle(dft_y), f, "DFT[y(t)] (phase domain)")
plot(
    np.real(inverse_discrete_fourier_transform(dft_y)) [:int(5/F * F_S)],
    t [:int(5/F * F_S)],
    "IDFT[DFT[y(t)]]"
)

In [ ]:
def fast_fourier_transform(
    x, i = 0, di = 1, *,
    pow: Literal[1]|Literal[-1] = 1):
    assert(abs(pow) == 1)
    
    if di == np.size(x): return x[i::di]

    a = fast_fourier_transform(x, i, di*2, pow=pow)
    b = fast_fourier_transform(x, i+di, di*2, pow=pow)  
    
    N = len(x)//di
    
    W = np.exp(-1j * 2*pi/N * pow)
    Wmat = np.pow(W, np.arange(N//2))
    return np.concatenate([a + b*Wmat, a - b*Wmat])

def inverse_fast_fourier_transform(x):
    return fast_fourier_transform(x, pow=-1) / len(x)

t = np.linspace(0, SAMPLE_COUNT / F_S, SAMPLE_COUNT)
f = np.linspace(0, F_S, SAMPLE_COUNT)

fft_x = lambda t: fast_fourier_transform(x(t))
plot(np.abs(fft_x(t)), f, "FFT[x(t)] (amplitude domain)")
plot(np.angle(fft_x(t)), f, "FFT[x(t)] (phase domain)")
plot(
    np.real(inverse_fast_fourier_transform(fft_x(t))) [:int(5/F * F_S)],
    t [:int(5/F * F_S)],
    "IFFT[FFT[x(t)]]"
)

fft_y = lambda t: fast_fourier_transform(y(t))
plot(np.abs(fft_y(t)), f, "FFT[y(t)] (amplitude domain)")
plot(np.angle(fft_y(t)), f, "FFT[y(t)] (phase domain)")
plot(
    np.real(inverse_fast_fourier_transform(fft_y(t))) [:int(5/F * F_S)],
    t [:int(5/F * F_S)],
    "IFFT[FFT[y(t)]]"
)


In [ ]:
def convolve(x, y, *, do_flip: bool = True):
    N = len(x)
    M = len(y)
    K = N+M-1
    
    x = np.concatenate([np.zeros(K-len(x)), x])
    y = np.concatenate([np.flip(y) if do_flip else y, np.zeros(K-len(y))])

    result = np.zeros(K)
    for i in range(len(result)):
        result[i] = sum(x*y)
        y = np.roll(y, 1)
    return result

def fast_convolve(x, y):
    N = len(x)
    M = len(y)
    K = N+M-1
    
    x = np.concatenate([np.zeros(K-len(x)), x])
    y = np.concatenate([np.zeros(K-len(y)), y])

    return np.fft.ifft(np.fft.fft(x) * np.fft.fft(y))
    
def correlate(x, y): return convolve(x, y, do_flip=False)
    
def fast_correlate(x, y):
    N = len(x)
    M = len(y)
    K = N+M-1
    
    x = np.concatenate([np.zeros(K-len(x)), x])
    y = np.concatenate([y, np.zeros(K-len(y))])

    return np.fft.ifft(np.fft.fft(x) * np.conj(np.fft.fft(y)))

t = np.linspace(0, 5 / F, SAMPLE_COUNT)
t_ = np.linspace(0, 10 / F, 2*SAMPLE_COUNT-1)

conv = lambda t: convolve(x(t), y(t))
conv_fft = lambda t: fast_convolve(x(t), y(t))
# fft_conv = lambda t: np.fft.fft(conv(t))
# tz = np.linspace(0, SAMPLE_COUNT / F_S, SAMPLE_COUNT)
# f = np.linspace(0, F_S*2, SAMPLE_COUNT*2-1)
plot(conv(t), t_, "Convolution")
plot(conv_fft(t), t_, "Convolution (through FFT)")
# plot(fft_conv(tz), f, "FFT[Convolution]")

corr = lambda t: correlate(x(t), y(t))
corr_fft = lambda t: fast_correlate(x(t), y(t))
plot(corr(t), t_, "Correlation")
plot(corr_fft(t), t_, "Correlation (through FFT)")

### С использованием библиотек

In [ ]:
t = np.linspace(0, 5 / F, SAMPLE_COUNT)
t_ = np.linspace(0, 10 / F, 2*SAMPLE_COUNT-1)

np_conv = lambda t: np.convolve(x(t), y(t))
plot(np_conv(t), t_, "NumPy Convolution")

np_corr = lambda t: np.correlate(x(t), y(t), "full")
plot(np_corr(t), t_, "NumPy Correlation")

In [ ]:
t = np.linspace(0, SAMPLE_COUNT / F_S, SAMPLE_COUNT)
f = np.linspace(0, F_S, SAMPLE_COUNT)

np_fft_x = lambda t: np.fft.fft(x(t))
plot(np.abs(np_fft_x(t)), f, "NumPy FFT[x(t)] (amplitude domain)")
plot(np.angle(np_fft_x(t)), f, "NumPy FFT[x(t)] (phase domain)")

np_fft_y = lambda t: np.fft.fft(y(t))
plot(np.abs(np_fft_y(t)), f, "NumPy FFT[y(t)] (amplitude domain)")
plot(np.angle(np_fft_y(t)), f, "NumPy FFT[y(t)] (phase domain)")